# Libraries

In [0]:
%pip install openpyxl
%restart_python

In [0]:
# Importing required libraries
import pandas as pd
import numpy as np

# Data

## Reading

In [0]:
raw_directory = '/Volumes/training_feature_development/marketing_mix_modelling/data_v1/raw_data/' # Change to the directory of the data
clean_directory = '/Volumes/training_feature_development/marketing_mix_modelling/data_v1/clean_data/' # Change to the directory of the data

# Reading raw data
sem = pd.read_excel(raw_directory + 'SEM EXH & LBE Mar 2023 to Jun 2025 weekly report_v2.xlsx', skiprows=2)
datorama = pd.read_excel(raw_directory + 'Datorama Mar 2023 to June 2025 daily report.xlsx', skiprows=4, skipfooter=1)
snapchat = pd.read_csv(raw_directory + 'Snapchat EXH & LBE Campaign & ad level Mar 2023 to Jun 2025.csv')
email = pd.read_csv(clean_directory + 'email.csv')
ooh = pd.read_csv(raw_directory + 'lbe_ooh.csv')
directbuy = pd.read_csv(raw_directory + 'lbe_directbuy.csv')
pr = pd.read_csv(clean_directory + 'pr.csv')
transactions = pd.read_csv(raw_directory + 'lbe_transactions.csv')

# Splitting Datorama dataset
meta = datorama[datorama['Publisher']=='Meta'].copy()
tiktok = datorama[datorama['Publisher']=='TikTok'].copy()
snapchat = snapchat[snapchat['Amount Spent']!=0]
programmatic = datorama[datorama['Channel']!='Social'].copy()

## Cleaning

In [0]:
# Defining week columns
sem['Week'] = pd.to_datetime(sem['Week']).dt.to_period('W-THU').dt.start_time
meta['week'] = pd.to_datetime(meta['Day']).dt.to_period('W-THU').dt.start_time
tiktok['week'] = pd.to_datetime(tiktok['Day']).dt.to_period('W-THU').dt.start_time
snapchat['week'] = pd.to_datetime(snapchat['Start Time']).dt.to_period('W-THU').dt.start_time
programmatic['week'] = pd.to_datetime(programmatic['Day']).dt.to_period('W-THU').dt.start_time
ooh['week'] = pd.to_datetime(ooh['week']).dt.to_period('W-THU').dt.start_time
directbuy['week'] = pd.to_datetime(directbuy['week']).dt.to_period('W-THU').dt.start_time
pr['week'] = pd.to_datetime(pr['week']).dt.to_period('W-THU').dt.start_time
transactions['week'] = pd.to_datetime(transactions['date']).dt.to_period('W-THU').dt.start_time

# Cleaning column names
for df in [sem, meta, tiktok, snapchat, programmatic, email, pr, transactions]:
    df.columns = df.columns.str.lower().str.replace(' ', '_')

In [0]:
# Filtering data frames
start_date, end_date = '2023-03-31', '2025-06-30'

sem = sem[(sem['account_name'].str.contains('LBE'))&(sem['week'].between(start_date, end_date))]
meta = meta[(meta['raw_campaign_name'].str.contains('LBE'))&(meta['week'].between(start_date, end_date))]
tiktok = tiktok[(tiktok['raw_campaign_name'].str.contains('LBE'))&(tiktok['week'].between(start_date, end_date))]
snapchat = snapchat[(snapchat['campaign_name'].str.contains('LBE'))&(snapchat['week'].between(start_date, end_date))]
programmatic = programmatic[(programmatic['raw_campaign_name'].str.contains('LBE'))&(programmatic['week'].between(start_date, end_date))]
ooh = ooh[ooh['week'].between(start_date, end_date)]
directbuy = directbuy[(directbuy['week'].between(start_date, end_date))]
pr = pr[(pr['week'].between(start_date, end_date))]
transactions =  transactions[(transactions['week'].between(start_date, end_date))]

# Feature Engineering

## Preprocessing

In [0]:
# Creating copies with only relevant columns
tmp1 = sem[['week', 'campaign', 'cost', 'impr.', 'clicks']].copy()
tmp2 = meta[['week', 'raw_campaign_name', 'investment', 'impressions', 'clicks']].copy()
tmp3 = tiktok[['week', 'raw_campaign_name', 'investment', 'impressions', 'clicks']].copy()
tmp4 = snapchat[['week', 'campaign_name', 'amount_spent', 'paid_impressions', 'clicks']].copy()
tmp5 = programmatic[['week', 'raw_campaign_name', 'investment', 'impressions', 'clicks']].copy()

# Creating a 'channel' column
cols = ['week', 'campaign_name', 'spend', 'impressions', 'clicks']
channels = ['SEM', 'Meta', 'TikTok', 'Snapchat', 'Programmatic']
for df in [tmp1, tmp2, tmp3, tmp4, tmp5]:
    df.columns = cols
    df['channel'] = channels.pop(0)

# Stacking all the data frames together
lbe_df = pd.concat([tmp1, tmp2, tmp3, tmp4, tmp5], ignore_index=True).fillna(0)

col = lbe_df['campaign_name']
lbe_df['location_type'] = np.select(
    [col.str.contains(r'Toronto & Mississauga|South & West|\+SQ'),
     col.str.contains('Toronto|South Ed|Calgary|Deerfoot|Winnipeg|Brentwood|Burnaby|Vancouver'), 
     col.str.contains('HPL|High'), col.str.contains('West Ed|Square|Miss|London|Mason|John|Avalon|Barrie'), 
     col.str.contains('LPL|Low'), col.str.contains('Royalmount|Granville|TRR_NL'), 
     col.str.contains('Brampton|Whitby|Dart'), col.str.contains('Fair|PDM_NL'),
     col.str.contains('TRR|Rec'), col.str.contains('PDM|Playdium')],
    ['Multiple', 'HPL', 'HPL', 'LPL', 'LPL', 
     'NL - TRR', 'PDM', 'NL - PDM', 
     'Canada - TRR', 'Canada - PDM'],
    default='Multiple'
)

# Stacking additional data frames together
lbe_df = pd.concat([lbe_df, ooh, directbuy], ignore_index=True)

In [0]:
# Cleaning the data frames
tmp1 = lbe_df[lbe_df['location_type']!='Multiple'].groupby(['week', 'location_type']).agg({'spend': 'sum'}).unstack().fillna(0).copy()
tmp1.columns = ['canada_pdm', 'canada_trr', 'hpl', 'lpl', 'nl_pdm', 'nl_trr', 'pdm']

tmp2 = transactions.groupby('week').agg({'transactions': 'sum', 'revenue': 'sum'}).copy()

tmp3 = pr.set_index('week').copy()
tmp3['pr_spend'] = tmp3[tmp3.columns[-3:-1]].sum(axis=1)
tmp3 = tmp3[['pr_spend']]

## Feature Creation

In [0]:
# Defining holidays
holidays_by_name = {
    "New Year's Day": ['2023-01-01', '2024-01-01', '2025-01-01'],
    'Family Day': ['2023-02-20', '2024-02-19', '2025-02-17'],
    'Good Friday': ['2023-04-07', '2024-03-29', '2025-04-18'],
    'Easter Monday': ['2023-04-10', '2024-04-01', '2025-04-21'],
    'Victoria Day': ['2023-05-22', '2024-05-20', '2025-05-19'],
    'Canada Day': ['2023-07-01', '2024-07-01', '2025-07-01'],
    'Labour Day': ['2023-09-04', '2024-09-02', '2025-09-01'],
    'Thanksgiving': ['2023-10-09', '2024-10-14', '2025-10-13'],
    'Christmas Day': ['2023-12-25', '2024-12-25', '2025-12-25']
}

holidays = [pd.to_datetime(date) for date_list in holidays_by_name.values() for date in date_list]
holidays = pd.Series(holidays).dt.to_period('W-THU').dt.start_time

# Creating new columns
tmp2['summer'] = np.where(tmp2.index.month.isin([7, 8]), 1, 0)
tmp2['holiday'] = np.where(tmp2.index.isin(holidays), 1, 0)
tmp2['week_num'] = np.arange(0, len(tmp2))
tmp2 = tmp2[['summer', 'holiday', 'week_num', 'transactions', 'revenue']]

In [0]:
df = pd.merge(tmp1, tmp3, how='left', left_index=True, right_index=True).fillna(0)
df = pd.merge(df, tmp2, how='right', left_index=True, right_index=True).fillna(0).reset_index()

In [0]:
# Reordering columns
spend_cols = list(df.columns[1:9])
season_cols = list(df.columns[[0]]) + list(df.columns[9:-2])
kpi_cols = list(df.columns[-2:])

df = df[season_cols + spend_cols + kpi_cols]
df.set_index('week', inplace=True)

In [0]:
# Exporting model data
directory = '/Volumes/training_feature_development/marketing_mix_modelling/data_v1/model_data/' # Change to the directory of the data

df.to_csv(directory + 'lbe_mmm.csv')